# Spatial-PCA modality benchmark

Compares MultiModalSCVI variants that differ **only** in how the spatial colocalization modality is represented:

1. `abundance_only`  — single-modality baseline (no spatial input)
2. `spatial_top500var` — existing baseline: top-500 highest-variance protein-pair features
3. `spatial_pca5`     — **new** option: PCA-reduce `spatial_asinh5` to 5 PCs (`build_spatial_pca_obsm`)
4. `spatial_pca20`    — **new** option: same, 20 PCs
5. `spatial_pca5_zscore` — same as #3 but z-scored before PCA

All variants use `loss_weights='auto'` so the modality-width difference (~500 vs 5 features) does not bias the ELBO.

Outputs:
- UMAPs of the joint latent (and per-modality where applicable) coloured by `cell_type_annot`, `condition`, `sample`
- Posterior predictive checks via `utils.plot_composite_ppc` for the abundance modality and the spatial modality
- Optional scIB-style metrics via `metrics.MultiModalVIMetrics`


In [ ]:
%load_ext autoreload
%autoreload 2

import sys
PIXELGEN_ROOT = '/home/projects/nyosef/zvise/PixelGen/PixelGen'
if PIXELGEN_ROOT not in sys.path:
    sys.path.append(PIXELGEN_ROOT)

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import scvi

from multimodalvi import MultiModalSCVI
from enums import AggMethod, D
from metrics import distr_autocorrelation_in_latent
from utils import (
    build_spatial_pca_obsm,
    plot_model_latents,
    plot_composite_ppc,
    get_dense,
    calculate_metrics,
)

scvi.settings.seed = 0
torch.set_float32_matmul_precision('high')
print('scvi-tools', scvi.__version__, '| CUDA', torch.cuda.is_available())

## 1. Load AnnData

`adata_all_annotated.h5ad` already ships with:
- `obsm['spatial_raw']` and `obsm['spatial_asinh5']` (12 395-pair DataFrames)
- `obsm['spatial_asinh5_top500var']` (top-variance baseline)
- abundance layers `arcsinh` / `clr` / `log1p`
- annotation column `cell_type_annot` and 15 `sample` batches


In [ ]:
ADATA_PATH = f'{PIXELGEN_ROOT}/New_Data/cache/adata_all_annotated.h5ad'
adata = sc.read_h5ad(ADATA_PATH)
print(adata)
print('layers:', list(adata.layers.keys()))
print('obsm keys:', list(adata.obsm.keys()))
print('cell_type_annot counts:')
print(adata.obs['cell_type_annot'].value_counts())

## 2. Build PCA-reduced spatial modalities (the new feature)

`build_spatial_pca_obsm` writes `adata.obsm[target_key]` (DataFrame) and `adata.uns[f'{target_key}_info']` (components + scaler stats) so it can be reused / inspected later.

In [ ]:
SPATIAL_SOURCE = 'spatial_asinh5'
TOP500_KEY     = 'spatial_asinh5_top500var'   # already in adata.obsm

key_pca5  = build_spatial_pca_obsm(adata, source_key=SPATIAL_SOURCE, n_components=5,  standardize='center')
key_pca20 = build_spatial_pca_obsm(adata, source_key=SPATIAL_SOURCE, n_components=20, standardize='center')
key_pca5z = build_spatial_pca_obsm(adata, source_key=SPATIAL_SOURCE, n_components=5,  standardize='zscore',
                                   target_key='spatial_asinh5_pca5_zscore')

for k in [key_pca5, key_pca20, key_pca5z]:
    info = adata.uns[f'{k}_info']
    print(f'  {k:40s}  shape={adata.obsm[k].shape}  '
          f"explained={info['explained_variance_ratio'].sum():.3f}  "
          f"standardize={info['standardize']}")

# Sanity: the model requires DataFrames in obsm.
for k in [key_pca5, key_pca20, key_pca5z]:
    assert isinstance(adata.obsm[k], pd.DataFrame), k
assert TOP500_KEY in adata.obsm, f'{TOP500_KEY} not in obsm'

## 3. Training helper

Same architecture / optimizer / epochs across runs — the only varying axis is the spatial obsm key (or its absence).

In [ ]:
ABUNDANCE_LAYER = 'arcsinh'
BATCH_KEY       = None

BASE_MODEL_KWARGS = dict(
    n_latent=20,
    n_hidden=128,
    n_layers=2,
    dropout_rate=0.1,
    loss_weights='auto',   # balances 5-PC vs 500-feat spatial vs 159-feat abundance
)
TRAIN_KWARGS = dict(max_epochs=200, batch_size=512, early_stopping=True)

def train_variant(spatial_key, name):
    """Returns (adata_view, model). Uses copies so each run owns its scvi registry."""
    a = adata.copy()
    n_mod = 1 if spatial_key is None else 2
    setup_kwargs = dict(layer=ABUNDANCE_LAYER, batch_key=BATCH_KEY, n_modalities=n_mod)
    if n_mod == 2:
        setup_kwargs['extra_modality_keys'] = [spatial_key]
    MultiModalSCVI.setup_anndata(a, **setup_kwargs)

    mk = dict(BASE_MODEL_KWARGS)
    mk['distrs'] = [D.Normal] if n_mod == 1 else [D.Normal, D.Normal]
    model = MultiModalSCVI(a, **mk)
    print(f'>>> training {name}: input_ds={model.input_ds}')
    model.train(**TRAIN_KWARGS)
    return a, model

## 4. Train all variants

In [ ]:
runs = {}
runs['abundance_only']     = train_variant(None,           'abundance_only')
runs['spatial_top500var']  = train_variant(TOP500_KEY,     'spatial_top500var')
runs['spatial_pca5']       = train_variant(key_pca5,       'spatial_pca5')
runs['spatial_pca20']      = train_variant(key_pca20,      'spatial_pca20')
runs['spatial_pca5_zscore']= train_variant(key_pca5z,      'spatial_pca5_zscore')

## 5. UMAPs of joint latent

`utils.plot_model_latents` produces a 3-panel grid per model (joint + 2 modalities). For the 1-modality run, only the joint panel is meaningful.


In [ ]:
models_only = {name: m for name, (_, m) in runs.items()}
for color in ['cell_type_annot', 'condition', 'sample']:
    print(f'=== UMAPs coloured by {color} ===')
    plot_model_latents(models_only, adata, color=color)

## 6. Posterior predictive checks

Mirrors `PBMSC/pbmsc_model_analysis.ipynb` — uses `model.get_normalized_expression(...)` and feeds observed/generated tensors into `utils.calculate_metrics`. Each run's spatial "ground truth" is the obsm key it was trained on (PCA-reduced for PCA models, top500var for the baseline).

In [ ]:
spatial_key_per_run = {
    'abundance_only':      None,
    'spatial_top500var':   TOP500_KEY,
    'spatial_pca5':        key_pca5,
    'spatial_pca20':       key_pca20,
    'spatial_pca5_zscore': key_pca5z,
}

all_metrics = []
for name, (a, model) in runs.items():
    imputed = model.get_normalized_expression(
        adata=a,
        return_mean_expression=True,
        return_l2_error=False,
        return_px_distrs=False,
        return_numpy=True,
    )

    # Abundance modality
    ab_obs = get_dense(a.layers[ABUNDANCE_LAYER])
    ab_gen = get_dense(imputed['exprs'][ABUNDANCE_LAYER])
    all_metrics.extend(calculate_metrics(ab_obs, ab_gen, name, 'Abundance'))

    # Spatial modality (skip for abundance_only)
    sp_key = spatial_key_per_run[name]
    if sp_key is not None:
        sp_obs = get_dense(a.obsm[sp_key])
        sp_gen = get_dense(imputed['exprs'][sp_key])
        all_metrics.extend(calculate_metrics(sp_obs, sp_gen, name, 'Spatial'))

metrics_df = pd.DataFrame(all_metrics)
metrics_df.head()

In [ ]:
model_names = list(runs.keys())
plot_composite_ppc('Abundance', metrics_df, scatter_color='cornflowerblue', model_names=model_names)
plot_composite_ppc('Spatial',   metrics_df, scatter_color='lightgreen',     model_names=model_names)

## 6.5 Genuine scVI single-modality baselines

Two standalone `scvi.model.SCVI` models (Normal likelihood, since `arcsinh`/`asinh` features are continuous and can be negative) trained on a single modality each:

- **abundance-only** — the `arcsinh` abundance layer (159 markers)
- **spatial-only** — the non-PCA `spatial_asinh5_top500var` rep (500 pairs)

Kept out of `runs` (different API from `MultiModalSCVI`); their latents are added to the autocorrelation benchmark in section 7 for context vs the multimodal joint latents.

In [ ]:
import anndata as ad

SCVI_KWARGS = dict(n_latent=20, n_hidden=128, n_layers=2, dropout_rate=0.1,
                   gene_likelihood='normal')   # continuous features

def train_scvi_baseline(X, var_names, name):
    """Single-modality scvi.model.SCVI baseline; returns the latent representation.

    scVI's machinery assumes non-negative, count-like input (it computes
    library = log(x.sum(1)) and log1p(x), and the decoder mean is softmax(.)*exp(library) >= 0).
    The spatial colocalization features are signed, so we shift each feature to >= 0
    before fitting (no-op for the already non-negative arcsinh abundance layer).
    The benchmark only needs the encoder latent z, so this shift is harmless.
    """
    Xd = np.asarray(get_dense(X), dtype='float32')
    Xd = np.clip(Xd - Xd.min(axis=0, keepdims=True), 0.0, None)   # per-feature shift to >= 0
    a = ad.AnnData(X=Xd, obs=adata.obs.copy())
    a.var_names = [str(v) for v in var_names]
    scvi.model.SCVI.setup_anndata(a, batch_key=BATCH_KEY)
    m = scvi.model.SCVI(a, **SCVI_KWARGS)
    print(f'>>> training vanilla scVI {name}: n_vars={a.n_vars}')
    m.train(**TRAIN_KWARGS)
    return m.get_latent_representation()

z_scvi_abundance = train_scvi_baseline(adata.layers[ABUNDANCE_LAYER], adata.var_names, 'abundance')
z_scvi_spatial   = train_scvi_baseline(adata.obsm[TOP500_KEY], adata.obsm[TOP500_KEY].columns, 'spatial')

## 7. Spatial autocorrelation benchmark

Mirrors `PBMSC/benchmark_autocorrelation.ipynb`: for each model's joint latent we build a kNN graph and compute Moran's I on each abundance and spatial feature. A latent that better organizes cells along biologically/spatially meaningful axes will produce *higher* Moran's I — features become smoother across neighbouring cells.

In [ ]:
# Store each model's joint latent on a single shared adata view so all latents live in one obsm.
adata_ac = adata.copy()
latent_keys, latent_names = [], []
for name, (a, model) in runs.items():
    z = model.get_latent_representation(a, modality='joint')
    key = f'z_joint__{name}'
    adata_ac.obsm[key] = z
    latent_keys.append(key)
    latent_names.append(name)

# --- extra latents for context ---
# spatial-modality latent of the basic non-pca multimodal model
a_b, m_b = runs['spatial_top500var']
adata_ac.obsm['z_spatial__spatial_top500var'] = m_b.get_latent_representation(a_b, modality=TOP500_KEY)
latent_keys.append('z_spatial__spatial_top500var')
latent_names.append('mmvi_top500var__spatialmod')

# genuine scVI single-modality baselines (from section 6.5)
adata_ac.obsm['z_scvi_abundance'] = z_scvi_abundance
latent_keys.append('z_scvi_abundance'); latent_names.append('scvi_abundance_only')
adata_ac.obsm['z_scvi_spatial'] = z_scvi_spatial
latent_keys.append('z_scvi_spatial'); latent_names.append('scvi_spatial_only')

print('latent obsm keys:', latent_keys)

In [ ]:
# Moran's I on abundance features, evaluated under each model's latent kNN graph.
# pca_kwargs caps PCA n_comps so it works for the 20-D joint latent.
autocorr_abundance = distr_autocorrelation_in_latent(
    adata_ac,
    latent_keys=latent_keys,
    names=latent_names,
    rep_key=ABUNDANCE_LAYER,
    pca_kwargs={'n_comps': 15},
)
print('abundance:', autocorr_abundance.shape)
autocorr_abundance.head()

In [ ]:
# Moran's I on spatial features (top500var, same ground-truth rep for every model).
autocorr_spatial = distr_autocorrelation_in_latent(
    adata_ac,
    latent_keys=latent_keys,
    names=latent_names,
    rep_key=TOP500_KEY,
    pca_kwargs={'n_comps': 15},
)
print('spatial:', autocorr_spatial.shape)
autocorr_spatial.head()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.histplot(data=autocorr_spatial, x='morans', hue='latent',
             kde=True, stat='density', common_norm=False, alpha=0.4, ax=axes[0])
axes[0].set_title(f"Spatial Moran's I  (rep={TOP500_KEY}, 500 pairs)")
axes[0].set_xlabel("Moran's I")

sns.histplot(data=autocorr_abundance, x='morans', hue='latent',
             kde=True, stat='density', common_norm=False, alpha=0.4, ax=axes[1])
axes[1].set_title(f"Abundance Moran's I  (rep={ABUNDANCE_LAYER}, {adata.obsm[ABUNDANCE_LAYER].shape[1]} markers)")
axes[1].set_xlabel("Moran's I")
plt.tight_layout(); plt.show()

In [ ]:
# Summary stats (mean / median / std of Moran's I across features) per model and feature type.
summary_rows = []
for label, df in [('Spatial', autocorr_spatial), ('Abundance', autocorr_abundance)]:
    for name in latent_names:
        s = df[df['latent'] == name]['morans']
        summary_rows.append({
            'Feature type': label, 'Model': name,
            'Mean': s.mean(), 'Median': s.median(), 'Std': s.std(),
        })
summary_df = pd.DataFrame(summary_rows)
summary_df.pivot(index='Model', columns='Feature type', values=['Mean', 'Median'])

In [ ]:
# Paired scatter: Moran's I per feature, abundance_only baseline vs each multimodal variant.
BASELINE = 'abundance_only'
comparisons = [name for name in latent_names if name != BASELINE]
fig, axes = plt.subplots(len(comparisons), 2, figsize=(11, 4.5 * len(comparisons)))
if len(comparisons) == 1:
    axes = axes.reshape(1, 2)
for row, name in enumerate(comparisons):
    for col, (label, df) in enumerate([('Spatial', autocorr_spatial), ('Abundance', autocorr_abundance)]):
        ax = axes[row, col]
        x = df[df['latent'] == BASELINE]['morans'].values
        y = df[df['latent'] == name]['morans'].values
        ax.scatter(x, y, alpha=0.4, s=12)
        lo = min(x.min(), y.min()); hi = max(x.max(), y.max())
        ax.plot([lo, hi], [lo, hi], 'k--', lw=1, alpha=0.5)
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi); ax.set_aspect('equal')
        ax.set_xlabel(f"Moran's I — {BASELINE}")
        ax.set_ylabel(f"Moran's I — {name}")
        n_above = int((y > x).sum()); n = len(x)
        ax.set_title(f'{label}   ({n_above}/{n} above diag)')
plt.tight_layout(); plt.show()

## 8. Optional: scIB-style benchmark via `MultiModalVIMetrics`

Compares biological conservation / batch correction across runs with a single shared `adata` view (uses each model's joint latent).

In [ ]:
try:
    from metrics import MultiModalVIMetrics
    mvi = MultiModalVIMetrics(
        adata,
        models=models_only,
        batch_key=BATCH_KEY,
        biological_key='cell_type_annot',
    )
    mvi.run()
    display(mvi.results if hasattr(mvi, 'results') else mvi)
except Exception as e:
    print('MultiModalVIMetrics step skipped:', repr(e))